# 作业2a：手搓多元线性回归

数据集是加州房价，有8个特征要预测房价。

线性回归就是假设y和x之间是线性关系：

$$\hat{y} = w_1 x_1 + w_2 x_2 + ... + w_n x_n + b$$

用梯度下降来训练，损失函数是均方误差：

$$J = \frac{1}{2m} \sum(\hat{y} - y)^2$$

梯度更新公式（这个我推了好几遍才推对，中间问AI才搞懂为什么损失函数要除以2m——原来是为了求导的时候把2消掉...）：

$$w = w - \alpha \frac{1}{m} X^T(\hat{y} - y)$$

$$b = b - \alpha \frac{1}{m} \sum(\hat{y} - y)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 1. 加载数据

In [ ]:
"""
加载加州房价数据集

运作流程：
    1. 从sklearn下载加州房价数据集
    2. 拿到特征矩阵X、目标值y和特征名称
    3. 打印数据集的基本信息看看长什么样

重要变量：
    - housing: 数据集对象，包含data、target、feature_names等
    - X: 特征矩阵，形状(20640, 8)，8个特征
    - y: 房价目标值，形状(20640,)
    - feature_names: 特征名称列表

依赖关系：
    - 依赖sklearn的fetch_california_housing函数
"""
housing = fetch_california_housing()
X = housing.data
y = housing.target
feature_names = housing.feature_names

print(f"数据集大小: {X.shape}")
print(f"特征: {feature_names}")
print(f"房价范围: [{y.min():.2f}, {y.max():.2f}]")

In [ ]:
"""
划分训练集和测试集，并对特征做标准化

运作流程：
    1. 用train_test_split把数据按8:2分成训练集和测试集
    2. 用StandardScaler对训练集做标准化（均值变0，方差变1）
    3. 测试集用训练集的scaler来变换（不能用测试集自己的！）
       （这个一开始也不懂，问AI才知道为什么要用训练集的参数来transform测试集，
        意思是测试集要假装没见过，只能用训练集的信息）

重要变量：
    - X_train, X_test: 训练和测试特征
    - y_train, y_test: 训练和测试目标值
    - scaler: StandardScaler对象，保存了训练集的均值和方差
    - X_train_scaled, X_test_scaled: 标准化后的特征

依赖关系：
    - 依赖sklearn的train_test_split和StandardScaler
    - 标准化后的数据才能喂给梯度下降训练
"""
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"训练集: {X_train_scaled.shape}")
print(f"测试集: {X_test_scaled.shape}")

## 2. 自己写线性回归

In [ ]:
class MyLinearRegression:
    """
    自己写的线性回归，用梯度下降训练
    
    属性：
        w: 权重向量
        b: 偏置
        lr: 学习率
        n_iters: 迭代次数
        losses: 记录每次迭代的损失
    """
    
    def __init__(self, learning_rate=0.01, n_iters=1000):
        self.w = None
        self.b = 0
        self.lr = learning_rate
        self.n_iters = n_iters
        self.losses = []
    
    def predict(self, X):
        """
        用训练好的权重和偏置来做预测
        
        运作流程：
            1. 把输入特征X和权重w做矩阵乘法，得到每个样本的加权求和
            2. 再加上偏置b，就得到最终的预测值
        
        重要变量：
            - X: 输入特征矩阵，形状是(m, n)，m是样本数，n是特征数（参数，局部）
            - self.w: 权重向量，形状是(n,)，每个特征对应一个权重（实例属性）
            - self.b: 偏置，一个标量，相当于截距（实例属性）
            - 返回值: 预测值y_hat，形状是(m,)（局部变量）
        
        依赖关系：
            - 需要self.w和self.b已经被训练好（先调用fit方法）
            - 依赖numpy的矩阵乘法运算(np.dot)
        """
        return np.dot(X, self.w) + self.b
    
    def compute_loss(self, y_hat, y):
        """
        计算均方误差损失，用来衡量预测值和真实值差多少
        
        运作流程：
            1. 算预测值和真实值的差（残差）
            2. 把残差平方，然后求和
            3. 除以2m得到最终损失（除以2是为了后面求导方便，2m和m只是差个常数倍）
        
        重要变量：
            - y_hat: 预测值，形状是(m,)
            - y: 真实值，形状是(m,)
            - m: 样本数量
            - diff: 预测值和真实值的差，形状是(m,)
            - 返回值: 标量损失值
        
        依赖关系：
            - 依赖numpy的求和和幂运算
            - 被fit方法调用，用来监控训练过程
        """
        m = len(y)
        diff = y_hat - y
        return np.sum(diff ** 2) / (2 * m)
    
    def fit(self, X, y):
        """
        用梯度下降法训练模型，找到最优的w和b
        
        运作流程：
            1. 初始化权重w为全0向量，偏置b为0
            2. 循环n_iters次迭代：
               a. 调用predict算出当前参数下的预测值y_hat（就是代入公式算预测值）
               b. 调用compute_loss算出当前损失，记录下来
               c. 算梯度：dw是损失对w的偏导，db是损失对b的偏导
               d. 沿梯度反方向更新w和b（这就是梯度下降）
            3. 返回训练好的self
        
        重要变量：
            - X: 训练特征矩阵，形状(m, n)，m是样本数，n是特征数（参数，局部）
            - y: 训练目标向量，形状(m,)（参数，局部）
            - m, n: 样本数和特征数（局部变量）
            - diff: 预测值和真实值的差，形状(m,)（局部变量）
            - dw: w的梯度，形状(n,)，告诉w该往哪个方向调（局部变量）
            - db: b的梯度，标量，告诉b该往哪个方向调（局部变量）
            - self.losses: 列表，记录每次迭代的损失值（实例属性）
        
        依赖关系：
            - 调用self.predict()算预测值
            - 调用self.compute_loss()计算损失
            - 依赖numpy的dot和sum运算
            - 训练前需要先对数据做标准化，否则梯度下降可能不收敛
        """
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0
        self.losses = []
        
        for i in range(self.n_iters):
            y_hat = self.predict(X)
            loss = self.compute_loss(y_hat, y)
            self.losses.append(loss)
            
            diff = y_hat - y
            dw = (1 / m) * np.dot(X.T, diff)
            db = (1 / m) * np.sum(diff)
            
            self.w = self.w - self.lr * dw
            self.b = self.b - self.lr * db
        
        return self

## 3. 训练

In [ ]:
"""
训练线性回归模型

运作流程：
    1. 创建MyLinearRegression实例，设置学习率0.01，迭代1000次
    2. 用标准化后的训练数据来fit模型
    3. 打印训练结果：权重、偏置、最终损失

重要变量：
    - model: 训练好的线性回归模型
    - model.w: 学到的8个权重
    - model.b: 学到的偏置
    - model.losses: 1000次迭代的损失记录

依赖关系：
    - 依赖前面定义的MyLinearRegression类
    - 依赖前面标准化好的X_train_scaled和y_train
"""
model = MyLinearRegression(learning_rate=0.01, n_iters=1000)
model.fit(X_train_scaled, y_train)

print("训练完了，看看结果")
print(f"权重: {model.w}")
print(f"偏置: {model.b:.4f}")
print(f"最终损失: {model.losses[-1]:.4f}")

## 4. 损失曲线

看看训练过程是不是在收敛

In [ ]:
"""
绘制训练损失曲线

运作流程：
    1. 把model.losses里记录的每次迭代损失画成折线图
    2. 如果曲线一直在下降说明模型在学习，如果平了说明收敛了
    3. 如果曲线震荡或者上升，说明学习率可能太大了

重要变量：
    - model.losses: 训练过程中记录的损失列表，长度等于迭代次数

依赖关系：
    - 依赖训练好的model（model.losses）
    - 依赖matplotlib的pyplot模块
"""
plt.figure(figsize=(8, 5))
plt.plot(range(len(model.losses)), model.losses, 'b-')
plt.xlabel('迭代次数')
plt.ylabel('损失')
plt.title('训练损失曲线')
plt.grid(True, alpha=0.3)
plt.show()

## 5. 评估

In [ ]:
"""
在测试集上评估模型效果

运作流程：
    1. 用训练好的模型对测试集做预测
    2. 计算MSE（均方误差）、RMSE（均方根误差）、MAE（平均绝对误差）
    3. 计算R²（决定系数），越接近1说明模型越好
       （R²这个指标一开始不知道，问AI才知道怎么算的）

重要变量：
    - y_pred: 模型在测试集上的预测值
    - mse: 均方误差，越小越好
    - rmse: 均方根误差，和y同量纲，好理解
    - mae: 平均绝对误差
    - r2: 决定系数，1表示完美预测

依赖关系：
    - 依赖训练好的model和标准化后的X_test_scaled
    - 依赖numpy的统计运算
"""
y_pred = model.predict(X_test_scaled)

mse = np.mean((y_pred - y_test) ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(y_pred - y_test))
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")

## 6. 可视化

In [ ]:
"""
绘制真实值vs预测值的散点图

运作流程：
    1. 把测试集的真实值和预测值画成散点图
    2. 画一条y=x的红色虚线作为参考线（如果预测完美，所有点都会落在这条线上）
    3. 点越靠近红线说明预测越准

重要变量：
    - y_test: 测试集真实房价
    - y_pred: 模型预测的房价

依赖关系：
    - 依赖前面评估cell中计算的y_pred
    - 依赖matplotlib的pyplot模块
"""
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.3, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('真实值')
plt.ylabel('预测值')
plt.title('真实值 vs 预测值')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
"""
绘制各特征的权重柱状图

运作流程：
    1. 把模型学到的8个权重画成柱状图，每个柱子对应一个特征
    2. 权重的绝对值越大，说明这个特征对房价的影响越大
    3. 权重为正说明特征越大房价越高，为负则相反

重要变量：
    - feature_names: 特征名称列表（比如MedInc、HouseAge等）
    - model.w: 模型学到的8个权重值

依赖关系：
    - 依赖训练好的model（model.w）
    - 依赖加载数据时得到的feature_names
    - 依赖matplotlib的pyplot模块
"""
plt.figure(figsize=(10, 5))
plt.bar(feature_names, model.w)
plt.xlabel('特征')
plt.ylabel('权重')
plt.title('各特征权重')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
"""
绘制残差分布图

运作流程：
    1. 计算残差 = 真实值 - 预测值
    2. 把残差和预测值画成散点图，加一条y=0的红色虚线
    3. 残差应该在0附近随机分布，如果有明显模式说明模型有问题

重要变量：
    - residuals: 残差（真实值减预测值），形状和y_test一样
    - y_pred: 预测值

依赖关系：
    - 依赖前面评估cell中计算的y_pred和y_test
    - 依赖matplotlib的pyplot模块
"""
residuals = y_test - y_pred
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.3, s=10)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('预测值')
plt.ylabel('残差')
plt.title('残差分布')
plt.grid(True, alpha=0.3)
plt.show()